# Baseball Broadcast Transcription Pipeline

This notebook transcribes old baseball radio broadcasts and produces a consolidated transcript for use in Retrosheet pitch sequence enrichment.

## Workflow
1. Configure game details
2. Verify GPU
3. Install dependencies
4. Mount Google Drive
5. Split MP3 into chunks
6. Transcribe each chunk with Whisper
7. Merge transcripts into a single consolidated file
8. Preview transcript

## Requirements
- MP3 file of the broadcast uploaded to Google Drive
- Google Colab with GPU runtime (Runtime → Change runtime type → T4 GPU)

## Important
Always run Step 1 first — all other steps depend on the variables defined there.

---

## Step 1: Configuration

Set your game details here. **This is the only cell you need to edit for each new game.**
Run this cell before any other step.

In [ ]:
# -------------------------------------------------------------------
# CONFIGURATION — edit these values for each game
# -------------------------------------------------------------------

# Retrosheet game ID (e.g. NYA197409250)
GAME_ID = "NYA197409250"

# Name of the MP3 file in your Google Drive folder (include extension)
# File names with spaces are fine
MP3_FILENAME = "1974 09-25 Red Sox at Yankees.mp3"

# Google Drive folder where your MP3 is stored and where output will be saved
DRIVE_FOLDER = "/content/drive/MyDrive/baseball_transcripts"

# Chunk size in seconds (1800 = 30 minutes)
# For ~3 hour games, 1800s gives ~6 chunks which Whisper handles well
CHUNK_SECONDS = 1800

# Whisper model to use
# large-v3 is the most accurate but slowest
# medium is ~4x faster but less accurate — try if large-v3 is too slow
WHISPER_MODEL = "large-v3"

# -------------------------------------------------------------------
# Imports and derived paths — no need to edit below this line
# -------------------------------------------------------------------
import os
import glob
import json
import subprocess
import time

MP3_PATH = os.path.join(DRIVE_FOLDER, MP3_FILENAME)
CHUNK_PREFIX = os.path.join(DRIVE_FOLDER, f"{GAME_ID}_chunk")
MERGED_OUTPUT = os.path.join(DRIVE_FOLDER, f"{GAME_ID}_FULL_TRANSCRIPT.json")
MERGED_TEXT_OUTPUT = os.path.join(DRIVE_FOLDER, f"{GAME_ID}_FULL_TRANSCRIPT.txt")

print(f"Game ID:        {GAME_ID}")
print(f"MP3 path:       {MP3_PATH}")
print(f"Chunk prefix:   {CHUNK_PREFIX}")
print(f"Merged output:  {MERGED_OUTPUT}")
print(f"Whisper model:  {WHISPER_MODEL}")
print("\n✅ Configuration loaded — ready to proceed")

## Step 2: Verify GPU

⚠️ **Before running this step**, make sure you have set the runtime to GPU:
1. Click **Runtime** in the top menu
2. Select **Change runtime type**
3. Set **Hardware accelerator** to **T4 GPU**
4. Click **Save** — the session will restart
5. Re-run Step 1, then continue here

If `nvidia-smi: command not found` appears below, you are on a CPU runtime and need to follow the steps above.

In [ ]:
import torch

!nvidia-smi

if torch.cuda.is_available():
    print(f"\n✅ GPU available: {torch.cuda.get_device_name(0)}")
else:
    print("\n⚠️  No GPU detected. Transcription will be much slower on CPU.")
    print("Go to Runtime → Change runtime type → T4 GPU")

## Step 3: Install Dependencies

In [ ]:
# Install Whisper
!pip install openai-whisper -q

# ffmpeg is usually pre-installed in Colab but verify
!apt-get install -y ffmpeg -q

print("✅ Dependencies installed")

## Step 4: Mount Google Drive

In [ ]:
# Guard: ensure Step 1 has been run
if 'MP3_PATH' not in dir():
    raise RuntimeError("❌ MP3_PATH not defined. Please run Step 1 first.")

from google.colab import drive
drive.mount('/content/drive')

# Verify the MP3 file exists
if os.path.exists(MP3_PATH):
    size_mb = os.path.getsize(MP3_PATH) / (1024 * 1024)
    print(f"✅ Found MP3: {MP3_FILENAME} ({size_mb:.1f} MB)")
else:
    print(f"❌ MP3 not found at: {MP3_PATH}")
    print("Check that the file is uploaded to your Drive folder and the filename matches exactly.")

## Step 5: Split MP3 into Chunks

Splits the broadcast into 30-minute chunks for more reliable Whisper transcription.
Output files will be named `{GAME_ID}_chunk_000.mp3`, `{GAME_ID}_chunk_001.mp3`, etc.

In [ ]:
# Guard: ensure Step 1 has been run
if 'MP3_PATH' not in dir():
    raise RuntimeError("❌ MP3_PATH not defined. Please run Step 1 first.")

# Split using ffmpeg
# -c copy means no re-encoding (fast, no quality loss)
cmd = [
    "ffmpeg",
    "-i", MP3_PATH,
    "-f", "segment",
    "-segment_time", str(CHUNK_SECONDS),
    "-c", "copy",
    "-y",  # overwrite existing files
    f"{CHUNK_PREFIX}_%03d.mp3"
]

result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0:
    chunks = sorted(glob.glob(f"{CHUNK_PREFIX}_*.mp3"))
    print(f"✅ Split into {len(chunks)} chunks:")
    for chunk in chunks:
        size_mb = os.path.getsize(chunk) / (1024 * 1024)
        print(f"   {os.path.basename(chunk)} ({size_mb:.1f} MB)")
else:
    print("❌ ffmpeg error:")
    print(result.stderr)

## Step 6: Transcribe Chunks with Whisper

Transcribes each chunk sequentially using the Whisper Python API.
On a T4 GPU, each 30-minute chunk takes roughly 3-5 minutes.

Output JSON files will be saved to your Drive folder alongside the MP3 chunks.

**Note:** If a chunk was already transcribed in a previous run, it will be skipped.

In [ ]:
# Guard: ensure Step 1 has been run
if 'CHUNK_PREFIX' not in dir():
    raise RuntimeError("❌ Configuration not loaded. Please run Step 1 first.")

import whisper

chunks = sorted(glob.glob(f"{CHUNK_PREFIX}_*.mp3"))

if not chunks:
    print("❌ No chunks found. Run Step 5 first.")
else:
    print(f"Found {len(chunks)} chunks to transcribe")
    print(f"Loading Whisper model: {WHISPER_MODEL}...")
    model = whisper.load_model(WHISPER_MODEL)
    print(f"✅ Model loaded\n")

    for i, chunk in enumerate(chunks):
        chunk_name = os.path.basename(chunk)
        json_path = chunk.replace(".mp3", ".json")

        # Skip if already transcribed
        if os.path.exists(json_path):
            print(f"⏭️  Skipping {chunk_name} (already transcribed)")
            continue

        print(f"🎙️  Transcribing chunk {i+1}/{len(chunks)}: {chunk_name}")
        start_time = time.time()

        result = model.transcribe(chunk, language="en")

        # Save output as JSON
        with open(json_path, "w") as f:
            json.dump(result, f, indent=2)

        elapsed = time.time() - start_time
        print(f"✅ Done in {elapsed/60:.1f} minutes — saved to {os.path.basename(json_path)}\n")

    print("\n✅ All chunks transcribed")

## Step 7: Merge Transcripts

Merges all chunk JSON files into a single consolidated transcript with continuous timestamps.
Also produces a plain text version for easy reading.

Output:
- `{GAME_ID}_FULL_TRANSCRIPT.json` — full transcript with timestamps (for LLM use)
- `{GAME_ID}_FULL_TRANSCRIPT.txt` — plain text version (for human review)

In [ ]:
# Guard: ensure Step 1 has been run
if 'CHUNK_PREFIX' not in dir():
    raise RuntimeError("❌ Configuration not loaded. Please run Step 1 first.")

chunk_jsons = sorted(glob.glob(f"{CHUNK_PREFIX}_*.json"))

if not chunk_jsons:
    print("❌ No transcript JSON files found. Run Step 6 first.")
else:
    print(f"Merging {len(chunk_jsons)} transcript files...\n")

    merged_segments = []

    for chunk_index, chunk_path in enumerate(chunk_jsons):
        with open(chunk_path) as f:
            data = json.load(f)

        chunk_segments = data.get("segments", [])

        # Use chunk index * CHUNK_SECONDS for reliable time offset
        # This avoids drift from Whisper ending segments slightly short of the boundary
        time_offset = chunk_index * CHUNK_SECONDS

        for segment in chunk_segments:
            merged_segments.append({
                "start": round(segment["start"] + time_offset, 2),
                "end": round(segment["end"] + time_offset, 2),
                "text": segment["text"]
            })

        print(f"   ✅ {os.path.basename(chunk_path)} — {len(chunk_segments)} segments (offset: {time_offset}s)")

    # Save merged JSON
    merged = {"game_id": GAME_ID, "segments": merged_segments}
    with open(MERGED_OUTPUT, "w") as f:
        json.dump(merged, f, indent=2)

    # Save plain text version
    with open(MERGED_TEXT_OUTPUT, "w") as f:
        f.write(f"Game ID: {GAME_ID}\n")
        f.write(f"Total segments: {len(merged_segments)}\n")
        f.write("-" * 60 + "\n\n")
        for seg in merged_segments:
            start_min = int(seg['start'] // 60)
            start_sec = seg['start'] % 60
            f.write(f"[{start_min:02d}:{start_sec:05.2f}]  {seg['text'].strip()}\n")

    print(f"\n✅ Merged transcript saved:")
    print(f"   JSON: {MERGED_OUTPUT}")
    print(f"   Text: {MERGED_TEXT_OUTPUT}")
    print(f"   Total segments: {len(merged_segments)}")

## Step 8: Preview Transcript

Print a section of the merged transcript to verify quality.
Adjust `start_time` and `end_time` (in seconds) to preview any part of the game.

In [ ]:
# Guard: ensure Step 1 has been run
if 'MERGED_OUTPUT' not in dir():
    raise RuntimeError("❌ Configuration not loaded. Please run Step 1 first.")

if not os.path.exists(MERGED_OUTPUT):
    print("❌ Merged transcript not found. Run Step 7 first.")
else:
    # Preview window in seconds — adjust as needed
    start_time = 0      # start of game
    end_time = 300      # first 5 minutes

    with open(MERGED_OUTPUT) as f:
        data = json.load(f)

    print(f"Game: {data['game_id']}")
    print(f"Previewing {start_time}s — {end_time}s\n")
    print("-" * 60)

    for seg in data["segments"]:
        if start_time <= seg["start"] <= end_time:
            start_min = int(seg['start'] // 60)
            start_sec = seg['start'] % 60
            print(f"[{start_min:02d}:{start_sec:05.2f}]  {seg['text'].strip()}")

---

## Notes & Future Improvements

### Whisper model caching
By default, Whisper re-downloads the `large-v3` model (~3GB) at the start of each new Colab session. This can be avoided by saving the model to Google Drive and loading from there. Low priority improvement but worth doing if running many games across multiple sessions.


The following parameters were not implemented in this pipeline but may improve transcription quality for old radio broadcasts. Test on a single chunk before applying to the full game.

**`condition_on_previous_text=False`**
Disables Whisper's use of previous output to condition the next segment. May help with degraded/noisy audio where the default conditioning can cause hallucinations.

**`temperature=0`**
Forces greedy (deterministic) decoding. More consistent output, generally better for transcription accuracy.

**`no_speech_threshold=0.6`**
Controls how aggressively Whisper marks segments as non-speech (outputs `...`). Raising it slightly may reduce missed pitch calls during crowd noise, but risks transcribing crowd noise as speech. Use with caution.

To test any of these, modify the `model.transcribe()` call in Step 6 like so:
```python
result = model.transcribe(
    chunk,
    language="en",
    condition_on_previous_text=False,
    temperature=0,
    no_speech_threshold=0.6
)
```

### Known transcription challenges for 1970s broadcasts
- **Player names** are frequently misheard (e.g. `Medich` → `Manage`, `Tiant` → `Tiong`)
- **Bunt calls** may be misheard (`bunts` → `busts`)
- **Crowd noise** causes `...` gaps in the transcript, potentially swallowing pitch calls
- **Spurious pitch calls** may appear from crowd noise being misinterpreted as speech

### Audio cleanup (optional)
For particularly poor quality recordings, consider preprocessing with Audacity or iZotope RX before transcribing:
1. Noise profile capture on a silent gap
2. Noise reduction
3. High-pass filter at ~120Hz to cut low hum
4. Normalize to -3dB

Based on testing, Whisper is robust enough that cleanup may not be necessary — transcribe first and clean up only if quality is insufficient.